# PIB — Step 1b: Tesseract OCR with fixed preprocessing

Same output as step 1 (word text, box, position in the page text), with a small preprocessing recipe that is
**chosen on a few `base` pages, frozen, and applied identically to all three splits**.

Options tested, in this order (each one is kept only if it helps on the tuning pages):

1. **Upscale 2x** (bicubic) before OCR — boxes stay in `[0, 1]` of the original page
2. **Page segmentation**: `--psm 3` (default) vs `--psm 11` (sparse text)
3. **Inverted second pass**: OCR the negative image and add the words that do not overlap any first-pass word
4. **Colour handling**: standard grayscale vs blue channel vs `max(R, G, B)`

Output: `base.jsonl`, `medium.jsonl`, `hard.jsonl` (one line per page, dataset order) and `meta.json` with the frozen settings.

```json
{"split": "base", "idx": 0, "source_pdf": "...", "page": 1, "total_pages": 1, "doc_type": "mri_reports",
 "width": 1190, "height": 1682,
 "text": "CENTRAL VALLEY CARDIOVASCULAR INSTITUTE\n1422 MEDICAL PLAZA WAY, ...",
 "words": [{"i": 0, "text": "CENTRAL", "bbox": [0.0612, 0.0971, 0.0731, 0.2122],
            "start": 0, "end": 7, "conf": 96.0, "line": 0, "pass": 1}, ...]}
```

**Change from step 1:** the page text is now built row by row (words grouped into visual lines, lines sorted
top-to-bottom, words left-to-right) instead of following Tesseract's block order. This is needed to merge the
two passes and to compare `--psm 3` with `--psm 11` on equal terms, and it keeps a field label next to its value.
`line` is the visual line index (use it to build one box per line); `pass` is 1 (normal) or 2 (inverted image).

In [ ]:
# 1. Install
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract datasets
!tesseract --version | head -1

In [ ]:
# 2. Settings
DATASET     = "disi-unibo-nlp/paint-it-black"
SPLITS      = ["base", "medium", "hard"]
OUT_DIR     = "/content/pib_ocr/tesseract_v2"
N_WORKERS   = 2                   # Colab has 2 CPU cores
MAX_PAGES   = None                # e.g. 5 for a quick test; None = all pages
SAVE_TO_DRIVE = False             # True = also copy the results to Google Drive at the end

# Tuning protocol: settings are chosen on a few pages of ONE split and then frozen.
TUNE_SPLIT      = "base"
N_TUNE_PER_TYPE = 2               # pages per document type (4 types -> 8 pages)

# Step 1 settings = starting point of the tuning
BASELINE = {"upscale": 1.0, "psm": 3, "invert_pass": False, "gray": "luma"}

# Options tried one after the other. The first value of each list is the current/simpler setting.
STAGES = [
    ("upscale",     [1.0, 2.0]),
    ("psm",         [3, 11]),
    ("invert_pass", [False, True]),
    ("gray",        ["luma", "blue", "max"]),
]
# On a tie the first listed value is kept, except for `gray`: base pages contain no coloured ink, so they can
# only show that a colour channel does no harm. If it does no harm, prefer "max", then "blue".
TIE_ORDER = {"gray": ["max", "blue", "luma"]}

# Step 1 results on all pages (for the before/after table at the end)
STEP1 = {"base":   {"located": 0.996, "exact": 0.713, "exact_trim": 0.897},
         "medium": {"located": 0.874, "exact": 0.518, "exact_trim": 0.669},
         "hard":   {"located": 0.811, "exact": 0.442, "exact_trim": 0.576}}

In [ ]:
# 3. Load the dataset
from datasets import load_dataset

try:
    dsd = load_dataset(DATASET)
except Exception as exc:
    # Private or gated repo: add your token as a Colab secret named HF_TOKEN (key icon on the left) and re-run.
    print(f"Anonymous load failed ({type(exc).__name__}); retrying with HF_TOKEN from Colab secrets...")
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    dsd = load_dataset(DATASET)

for s in SPLITS:
    print(f"{s:7s} {len(dsd[s]):4d} pages")

In [ ]:
# 4. OCR functions
import os
os.environ["OMP_THREAD_LIMIT"] = "1"   # one thread per Tesseract process; we parallelise over pages instead

import json, time, statistics
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pytesseract
from pytesseract import Output
from PIL import Image, ImageChops, ImageOps


def to_gray(img, mode):
    """RGB page -> single-channel image given to Tesseract."""
    img = img.convert("RGB")
    if mode == "luma":                      # standard grayscale conversion
        return img.convert("L")
    if mode == "blue":                      # blue channel: blue ink turns light, black text stays black
        return img.getchannel("B")
    if mode == "max":                       # max(R, G, B): any saturated colour turns light
        r, g, b = img.split()
        return ImageChops.lighter(ImageChops.lighter(r, g), b)
    raise ValueError(f"unknown gray mode: {mode}")


def tesseract_words(gray, psm, lang="eng"):
    """One Tesseract pass. Returns word dicts with boxes normalised by the size of `gray`."""
    W, H = gray.size
    # Output.DICT keeps every token as a string (a DataFrame would turn words like "NA" into NaN).
    d = pytesseract.image_to_data(gray, lang=lang, config=f"--oem 1 --psm {psm}", output_type=Output.DICT)
    words = []
    for k in range(len(d["text"])):
        tok = (d["text"][k] or "").strip()
        if d["level"][k] != 5 or not tok:   # 5 = word level
            continue
        left, top, w, h = d["left"][k], d["top"][k], d["width"][k], d["height"][k]
        clip = lambda v: round(min(1.0, max(0.0, v)), 5)
        words.append({"text": tok, "conf": float(d["conf"][k]),
                      "bbox": [clip(top / H), clip(left / W), clip((top + h) / H), clip((left + w) / W)]})
    return words


def _boxes_overlap(a, b):
    """True if two [y0, x0, y1, x1] boxes share any area."""
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])


def build_lines(words):
    """Group words into visual lines and order them top-to-bottom, left-to-right.

    Words are visited left to right. A word joins the nearest line whose last "real" word
    (one containing a letter or digit) it overlaps vertically by at least half of the smaller height;
    following the line word by word tolerates a slight skew. Two real words must also have similar heights (ratio <= 3),
    so an oversized noise box cannot glue several lines together. Punctuation-only tokens join by
    overlap alone and are never used as the reference for the next word.
    """
    height = lambda w: max(1e-9, w["bbox"][2] - w["bbox"][0])
    real = lambda w: any(ch.isalnum() for ch in w["text"])
    lines = []                                             # each: {"words": [...], "ref": word}
    for w in sorted(words, key=lambda w: (w["bbox"][1], w["bbox"][0])):
        best, best_key = None, None
        for ln in lines:
            ref = ln["ref"]
            ov = (min(w["bbox"][2], ref["bbox"][2]) - max(w["bbox"][0], ref["bbox"][0])) / min(height(w), height(ref))
            similar = max(height(w), height(ref)) <= 3 * min(height(w), height(ref))
            if ov >= 0.5 and (similar or not real(w) or not real(ref)):
                key = (abs(w["bbox"][1] - ref["bbox"][3]), -ov)  # nearest line end, then best overlap
                if best is None or key < best_key:
                    best, best_key = ln, key
        if best is None:
            lines.append({"words": [w], "ref": w})
        else:
            best["words"].append(w)
            if real(w) or not real(best["ref"]):
                best["ref"] = w
    y_mid = lambda ln: statistics.median((w["bbox"][0] + w["bbox"][2]) / 2 for w in ln["words"])
    lines.sort(key=lambda ln: (y_mid(ln), ln["words"][0]["bbox"][1]))
    return [ln["words"] for ln in lines]


def ocr_page(img, cfg):
    """OCR one PIL page with the settings in `cfg`.

    Returns (text, words, (width, height)). Words are in reading order; each has its text,
    a [y_min, x_min, y_max, x_max] box in [0, 1], [start, end) offsets into `text`,
    its line index and the pass that produced it (1 = normal, 2 = inverted image).
    """
    orig_size = img.size
    gray = to_gray(img, cfg["gray"])
    if cfg["upscale"] != 1.0:
        gray = gray.resize((round(gray.width * cfg["upscale"]), round(gray.height * cfg["upscale"])), Image.BICUBIC)
    # Boxes are divided by the size of the image Tesseract saw, so they are already
    # in [0, 1] of the original page whatever the upscale factor.
    words = tesseract_words(gray, cfg["psm"])
    for w in words:
        w["pass"] = 1
    if cfg["invert_pass"]:
        first = list(words)
        for w in tesseract_words(ImageOps.invert(gray), cfg["psm"]):
            if not any(_boxes_overlap(w["bbox"], f["bbox"]) for f in first):
                w["pass"] = 2
                words.append(w)

    out, parts, pos = [], [], 0
    for li, line in enumerate(build_lines(words)):
        for wi, w in enumerate(line):
            sep = "" if not out else (" " if wi else "\n")
            parts.append(sep + w["text"])
            pos += len(sep)
            out.append({"i": len(out), "text": w["text"], "bbox": w["bbox"], "start": pos,
                        "end": pos + len(w["text"]), "conf": w["conf"], "line": li, "pass": w["pass"]})
            pos += len(w["text"])
    return "".join(parts), out, orig_size


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

In [ ]:
# 5. Ground-truth check helpers and tuning pages
import string


def _as_records(anns):
    """Annotations as a list of dicts (handles both list-of-dicts and dict-of-lists layouts)."""
    if isinstance(anns, dict):
        keys = list(anns)
        return [dict(zip(keys, vals)) for vals in zip(*(anns[k] for k in keys))]
    return anns or []


def words_in_boxes(words, bboxes, tol=0.002):
    """OCR words whose centre lies inside one of the ground-truth boxes."""
    hits = []
    for w in words:
        cy, cx = (w["bbox"][0] + w["bbox"][2]) / 2, (w["bbox"][1] + w["bbox"][3]) / 2
        if any(b[0] - tol <= cy <= b[2] + tol and b[1] - tol <= cx <= b[3] + tol for b in bboxes):
            hits.append(w)
    return hits


_norm = lambda t: " ".join(t.lower().split())
_trim = lambda t: " ".join(tok.strip(string.punctuation) for tok in _norm(t).split()).strip()


def gt_counts(words, annotations):
    """[entities, located, exact text, exact text ignoring edge punctuation] for one page."""
    c = [0, 0, 0, 0]
    for ann in _as_records(annotations):
        hits = words_in_boxes(words, ann["bboxes"])
        got = " ".join(w["text"] for w in hits)
        c[0] += 1
        c[1] += bool(hits)
        c[2] += _norm(got) == _norm(ann["text"])
        c[3] += _trim(got) == _trim(ann["text"])
    return c


# Tuning pages: N_TUNE_PER_TYPE pages per document type, evenly spaced inside each type (deterministic).
doc_types = list(dsd[TUNE_SPLIT]["doc_type"])
TUNE_PAGES = []
for dt in sorted(set(doc_types)):
    idxs = [i for i, d in enumerate(doc_types) if d == dt]
    for k in range(N_TUNE_PER_TYPE):
        TUNE_PAGES.append(idxs[(k + 1) * len(idxs) // (N_TUNE_PER_TYPE + 1)])
TUNE_PAGES = sorted(set(TUNE_PAGES))
print(f"Tuning on {len(TUNE_PAGES)} pages of '{TUNE_SPLIT}':", [(i, doc_types[i]) for i in TUNE_PAGES])

tune_rows = [dsd[TUNE_SPLIT][i] for i in TUNE_PAGES]          # images + annotations, loaded once
_tune_cache = {}


def eval_config(cfg):
    """Run one configuration on the tuning pages and compare with the ground truth."""
    key = json.dumps(cfg, sort_keys=True)
    if key not in _tune_cache:
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
            results = list(pool.map(lambda r: ocr_page(r["image"], cfg), tune_rows))
        tot, n_words, confs = [0, 0, 0, 0], 0, []
        for row, (_, words, _) in zip(tune_rows, results):
            tot = [a + b for a, b in zip(tot, gt_counts(words, row["annotations"]))]
            n_words += len(words)
            confs += [w["conf"] for w in words]
        n = max(1, tot[0])
        _tune_cache[key] = {
            "entities": tot[0], "located": tot[1] / n, "exact": tot[2] / n, "exact_trim": tot[3] / n,
            "words_per_page": n_words / len(tune_rows), "mean_conf": sum(confs) / max(1, len(confs)),
            "pass2_words": sum(w["pass"] == 2 for _, ws, _ in results for w in ws),
            "sec_per_page": (time.time() - t0) / len(tune_rows) * N_WORKERS,
        }
    return _tune_cache[key]

In [ ]:
# 6. Staged tuning on the base tuning pages (a few minutes)
#    Selection score: exact text ignoring edge punctuation, then located, then strict exact text.
score = lambda m: (round(m["exact_trim"], 6), round(m["located"], 6), round(m["exact"], 6))

current = dict(BASELINE)
tuning_log = []
print(f"{'setting':22s} {'located':>8s} {'exact':>7s} {'exact*':>7s} {'words/pg':>9s} {'conf':>6s} {'pass2':>6s} {'s/page':>7s}   decision")
m = eval_config(current)
print(f"{'baseline (step 1)':22s} {m['located']:8.1%} {m['exact']:7.1%} {m['exact_trim']:7.1%} "
      f"{m['words_per_page']:9.1f} {m['mean_conf']:6.1f} {m['pass2_words']:6d} {m['sec_per_page']:7.1f}")
tuning_log.append({"config": dict(current), **m, "decision": "baseline"})

for name, values in STAGES:
    results = {v: eval_config({**current, name: v}) for v in values}
    best_score = max(score(r) for r in results.values())
    order = TIE_ORDER.get(name, values)
    chosen = next(v for v in order if score(results[v]) == best_score)
    for v in values:
        if v == current[name]:
            continue
        r = results[v]
        decision = "KEEP" if v == chosen else "no"
        print(f"{name + ' = ' + str(v):22s} {r['located']:8.1%} {r['exact']:7.1%} {r['exact_trim']:7.1%} "
              f"{r['words_per_page']:9.1f} {r['mean_conf']:6.1f} {r['pass2_words']:6d} {r['sec_per_page']:7.1f}   {decision}")
        tuning_log.append({"config": {**current, name: v}, **r, "decision": decision})
    current[name] = chosen

SUGGESTED = dict(current)
print("\nexact* = exact text ignoring punctuation at the edges of each word")
print("Suggested settings:", SUGGESTED)

In [ ]:
# 7. Freeze the settings (same for every split). Edit here only if you and your mentor decide otherwise.
FROZEN = dict(SUGGESTED)
# FROZEN["gray"] = "blue"        # example of a manual override
print("FROZEN =", FROZEN)

In [ ]:
# 8. Run OCR on every split with the frozen settings
def ocr_split(ds, split, cfg, out_dir=OUT_DIR, max_pages=MAX_PAGES, n_workers=N_WORKERS):
    """OCR one split and append one JSON line per page. Safe to re-run: finished pages are skipped."""
    out_path = Path(out_dir) / f"{split}.jsonl"
    n = len(ds) if max_pages is None else min(max_pages, len(ds))
    n_done = len(read_jsonl(out_path)) if out_path.exists() else 0   # pages are written in order
    if n_done >= n:
        print(f"[{split}] already complete ({n_done} pages) -> {out_path}")
        return out_path

    def one(i):
        row = ds[i]
        text, words, (width, height) = ocr_page(row["image"], cfg)
        return {"split": split, "idx": i, "source_pdf": row.get("source_pdf"), "page": row.get("page"),
                "total_pages": row.get("total_pages"), "doc_type": row.get("doc_type"),
                "width": width, "height": height, "text": text, "words": words}

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as pool, open(out_path, "a", encoding="utf-8") as f:
        for start in range(n_done, n, 8):                              # small batches keep memory low
            idxs = list(range(start, min(start + 8, n)))
            for rec in pool.map(one, idxs):
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            done = idxs[-1] + 1
            print(f"\r[{split}] {done}/{n} pages  ({(time.time() - t0) / (done - n_done):.1f} s/page)", end="")
    print(f"\n[{split}] saved -> {out_path}")
    return out_path


meta = {
    "dataset": DATASET, "splits": SPLITS, "engine": "tesseract",
    "tesseract_version": str(pytesseract.get_tesseract_version()), "lang": "eng", "oem": 1,
    "settings": FROZEN,
    "settings_help": {
        "upscale": "bicubic upscaling factor applied before OCR",
        "psm": "Tesseract page segmentation mode",
        "invert_pass": "second pass on the inverted image; its words are added only if they overlap no first-pass word",
        "gray": "luma = standard grayscale, blue = blue channel, max = max(R, G, B)",
    },
    "tuning": {"split": TUNE_SPLIT, "pages": TUNE_PAGES, "baseline": BASELINE,
               "score": "exact text ignoring edge punctuation, then located, then exact text", "log": tuning_log},
    "bbox_format": "[y_min, x_min, y_max, x_max], normalised to [0, 1] by page height (y) and width (x)",
    "offsets": "page['text'][word['start']:word['end']] == word['text']",
    "text_order": "visual lines top-to-bottom, words left-to-right; ' ' between words, '\\n' between lines",
}

# Guard: never mix pages produced with different settings in the same folder.
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
meta_path = Path(OUT_DIR) / "meta.json"
if meta_path.exists() and json.loads(meta_path.read_text())["settings"] != FROZEN:
    raise RuntimeError(f"{OUT_DIR} already holds results made with other settings. "
                       f"Delete that folder or change OUT_DIR, then re-run this cell.")
meta_path.write_text(json.dumps(meta, indent=2))

out_paths = {s: ocr_split(dsd[s], s, FROZEN) for s in SPLITS}

In [ ]:
# 9. Integrity checks on the saved files + summary
ocr = {s: read_jsonl(out_paths[s]) for s in SPLITS}

for s in SPLITS:
    pages = ocr[s]
    n_expected = len(dsd[s]) if MAX_PAGES is None else min(MAX_PAGES, len(dsd[s]))
    assert len(pages) == n_expected, f"{s}: {len(pages)} pages saved, expected {n_expected}"
    assert [p["idx"] for p in pages] == list(range(len(pages))), f"{s}: pages are not in dataset order"
    for p in pages:
        for w in p["words"]:
            assert p["text"][w["start"]:w["end"]] == w["text"], (s, p["idx"], w)
            y0, x0, y1, x1 = w["bbox"]
            assert 0 <= y0 <= y1 <= 1 and 0 <= x0 <= x1 <= 1, (s, p["idx"], w)
    n_words = sum(len(p["words"]) for p in pages)
    confs = [w["conf"] for p in pages for w in p["words"]]
    print(f"{s:7s} pages={len(pages):4d}  words={n_words:6d}  words/page={n_words / len(pages):6.1f}  "
          f"mean conf={sum(confs) / max(1, len(confs)):5.1f}  conf<50: {sum(c < 50 for c in confs):5d}  "
          f"from inverted pass: {sum(w['pass'] == 2 for p in pages for w in p['words']):5d}  "
          f"pages with no words: {[p['idx'] for p in pages if not p['words']]}")
print("Offsets and box ranges are consistent on every page.")

In [ ]:
# 10. Check against the ground truth on ALL pages, compared with step 1
from collections import defaultdict

final = {}
per_label = defaultdict(lambda: defaultdict(lambda: [0, 0]))        # label -> split -> [entities, located]
for s in SPLITS:
    gt_rows = dsd[s].select_columns(["annotations"])                # skips image decoding
    tot = [0, 0, 0, 0]
    for page in ocr[s]:
        anns = _as_records(gt_rows[page["idx"]]["annotations"])
        tot = [a + b for a, b in zip(tot, gt_counts(page["words"], anns))]
        for ann in anns:
            c = per_label[ann["label"]][s]
            c[0] += 1
            c[1] += bool(words_in_boxes(page["words"], ann["bboxes"]))
    final[s] = {"entities": tot[0], "located": tot[1] / tot[0], "exact": tot[2] / tot[0], "exact_trim": tot[3] / tot[0]}

print(f"{'split':7s} {'entities':>8s}   {'located':>17s}   {'exact text':>17s}   {'exact, edge punct. ignored':>27s}")
for s in SPLITS:
    cols = "   ".join(f"{STEP1[s][k]:6.1%} -> {final[s][k]:6.1%}" + ("          " if k == "exact_trim" else "")
                      for k in ("located", "exact", "exact_trim"))
    print(f"{s:7s} {final[s]['entities']:8d}   {cols}")
print("(step 1 -> this notebook)\n")

print(f"{'located, by label':24s} {'n':>5s} " + " ".join(f"{s:>7s}" for s in SPLITS))
for label, per in sorted(per_label.items(), key=lambda kv: kv[1][SPLITS[-1]][1] / kv[1][SPLITS[-1]][0]):
    print(f"{label:24s} {per[SPLITS[0]][0]:5d} " + " ".join(f"{per[s][1] / per[s][0]:7.1%}" for s in SPLITS))

meta["gt_check"] = final
meta_path.write_text(json.dumps(meta, indent=2))

In [ ]:
# 11. Visual check: draw the word boxes on 3 pages
import matplotlib.pyplot as plt
from PIL import ImageDraw

CHECK = [("base", 0), ("medium", 0), ("hard", 0)]   # (split, page index) - change freely

check_dir = Path(OUT_DIR) / "check"
check_dir.mkdir(parents=True, exist_ok=True)
for split, idx in CHECK:
    page = ocr[split][idx]
    vis = dsd[split][idx]["image"].convert("RGB").copy()
    W, H = vis.size
    draw = ImageDraw.Draw(vis)
    for w in page["words"]:
        y0, x0, y1, x1 = w["bbox"]
        colour = (0, 90, 255) if w["pass"] == 2 else ((230, 120, 0) if w["conf"] < 50 else (220, 0, 0))
        draw.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=colour, width=1)
    vis.save(check_dir / f"{split}_{idx:04d}.png")
    plt.figure(figsize=(12, 17))
    plt.imshow(vis)
    plt.axis("off")
    plt.title(f"{split} / page {idx}  -  {len(page['words'])} words "
              f"(red = word, orange = confidence < 50, blue = from inverted pass)")
    plt.show()
    print(page["text"][:600], "\n" + "-" * 80)

In [ ]:
# 12. Settings to report + package the results
import shutil

gray_txt = {"luma": "standard grayscale conversion", "blue": "blue channel only", "max": "per-pixel max(R, G, B)"}
print("OCR settings (identical for base, medium and hard):")
print(f"  Tesseract {meta['tesseract_version']}, English, LSTM engine (--oem 1), --psm {FROZEN['psm']}")
print(f"  input: {gray_txt[FROZEN['gray']]}, " +
      (f"upscaled {FROZEN['upscale']:g}x with bicubic interpolation" if FROZEN["upscale"] != 1 else "no upscaling"))
print("  second pass on the inverted image: " + ("yes (non-overlapping words added)" if FROZEN["invert_pass"] else "no"))
print(f"  chosen on {len(TUNE_PAGES)} pages of the {TUNE_SPLIT} split (indices {TUNE_PAGES}), then frozen")

zip_path = shutil.make_archive("/content/pib_ocr_tesseract_v2", "zip", OUT_DIR)
print("\nZipped:", zip_path)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/pib_ocr/tesseract_v2"
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print("Copied to:", dest)
else:
    from google.colab import files
    files.download(zip_path)

## Reading the file in the next steps

```python
import json
pages = [json.loads(l) for l in open("/content/pib_ocr/tesseract_v2/base.jsonl", encoding="utf-8")]
page = pages[0]
page["text"]                      # full page text (input for GLiNER / LLM taggers)
w = page["words"][0]
page["text"][w["start"]:w["end"]] # == w["text"]
w["bbox"]                         # [y_min, x_min, y_max, x_max] in [0, 1]
```

To turn a predicted character span `[a, b)` into boxes: take the words with `start < b and end > a`,
group them by `line`, and take the enclosing rectangle of each group (one box per line).
Take the entity text from `page["text"][a:b]`, not from the whole words.